In [95]:
# import packages
import pandas as pd
import numpy as np
import os
import requests
import plotly.express as px

os.environ["PUDL_DATA_STORE"] = "s3"
os.environ["PUDL_BUILD"] = "stable"

%reload_ext autoreload
%autoreload 2

year = 2025

## Scope 2 Analysis
How important is consumption-based: role of imports across regions. Look at gross imports
How important is spatial granularity: look at BA, region, national

### Question 1: role of imports
How much of regional load is generally served by imports
Data/methods:
1. download INTERCHANGE data
2. Calculate gross imports and gross exports for each region
3. Group by BA-year
4. Merge with balance data
5. % imports = imports_mwh / (demand_mwh + exports_mwh)

This could be evaluated from an emissions perspective as well:
CO2 imported / (CO2 consumed + CO2 exported)

### Question 2: Hierarchy ordering
How should spatial granularity / temporal granularity / type be prioritized. 
We want to calculate emission rates based on each method, then calculate total annual emissions by multiplying the EF by the demand in each hour and summing
We can then calculate an annual (and hourly) error rate, treating the hourly/BA/consumed factors as "truth"

Start with a single BA (MISO), then expand analysis to all

Data methods:
1. Download the BA file, filtering to a specific year of data
2. Add month/year and region columns
3. Calculate rollups: sum emissions and MWh and use to recalculate and transform to hourly
4. Calculate interconnection factors - need to sum across all BAs/regions in an interconnection, and roll these up
5. Get national factors for reference (US48 file)
6. Calculate hourly emissions by multiplying demand by EF
7. Calculate error rates
8. Get annual average error metrics, separated by category
9. Sort by different ranking and see how error rates decline 


How to compare: 
- For each BA-hour, calculate the following factors
    - BA_hour_consumed
    - ba_hour_produced
    - ba_month_consumed
    - ba_month_produced
    - ba_year_consumed
    - ba_year_produced
    - grid_hour_consumed
    - grid_hour_produced
    - grid_month_consumed
    - grid_month_produced
    - grid_year_consumed
    - grid_year_produced
    - national_hour_consumed
    - national_hour_produced
    - national_month_consumed
    - national_month_produced
    - national_year_consumed
    - national_year_produced

Spatial aggregations:
- BA: operational grid boundary
- Region: could be example of large vs small BAs
- Interconnection: grid-wide
- National: example of today


### Question 3: importance of local regions
Do "local" subdivisions provide meaningful gain in accuracy. 

Test 1: MISO
- Use data from MISO dashboard for LRZs, and LRZ demand data from EIA-930
- Use that as the source of truth, and compare to MISO-wide numbers

Test 2: EIA "regions" as proxy for large regions
- Examples: Northwest, Florida, Southwest, Carolinas



In [ ]:
# region and columns
ba_codes = [
    "AECI",
    "AVA",
    "AVRN",
    "AZPS",
    "BANC",
    "BHBA",
    "BPAT",
    "CHPD",
    "CISO",
    "CPLE",
    "CPLW",
    "DEAA",
    "DOPD",
    "DUK",
    "EEI",
    "EPE",
    "ERCO",
    "FMPP",
    "FPC",
    "FPL",
    "GCPD",
    "GLHB",
    "GRID",
    "GRIF",
    "GRMA",
    "GVL",
    "GWA",
    "HGMA",
    "HST",
    "IID",
    "IPCO",
    "ISNE",
    "JEA",
    "LDWP",
    "LGEE",
    "MISO",
    "NEVP",
    "NSB",
    "NWMT",
    "NYIS",
    "OVEC",
    "PACE",
    "PACW",
    "PGE",
    "PJM",
    "PNM",
    "PSCO",
    "PSEI",
    "SC",
    "SCEG",
    "SCL",
    "SEC",
    "SEPA",
    "SIKE",
    "SOCO",
    "SPA",
    "SRP",
    "SWPP",
    "SWPW",
    "TAL",
    "TEC",
    "TEPC",
    "TIDC",
    "TPWR",
    "TVA",
    "WACM",
    "WALC",
    "WAUE",
    "WAUW",
    "WWA",
    "YAD",
    "AESO",
    "BCHA",
    "HQT",
    "IESO",
    "MHEB",
    "NBSO",
    "SPC",
    "CEN",
    "CFE",
]

region_codes = [
    "CAL",
    "CAR",
    "CENT",
    "FLA",
    "MIDA",
    "MIDW",
    "NE",
    "NW",
    "NY",
    "SE",
    "SW",
    "TEN",
    "TEX",
    "US48",
    "CAN",
    "MEX",
]

ba_to_region = {
    "AEC": "SE",
    "AECI": "MIDW",
    "AVA": "NW",
    "AVRN": "NW",
    "AZPS": "SW",
    "BANC": "CAL",
    "BHBA": "NW",
    "BPAT": "NW",
    "CHPD": "NW",
    "CISO": "CAL",
    "CPLE": "CAR",
    "CPLW": "CAR",
    "DEAA": "SW",
    "DOPD": "NW",
    "DUK": "CAR",
    "EEI": "MIDW",
    "EPE": "SW",
    "ERCO": "TEX",
    "FMPP": "FLA",
    "FPC": "FLA",
    "FPL": "FLA",
    "GCPD": "NW",
    "GLHB": "MIDW",
    "GRID": "NW",
    "GRIF": "SW",
    "GRMA": "SW",
    "GVL": "FLA",
    "GWA": "NW",
    "HGMA": "SW",
    "HST": "FLA",
    "IID": "CAL",
    "IPCO": "NW",
    "ISNE": "NE",
    "JEA": "FLA",
    "LDWP": "CAL",
    "LGEE": "MIDW",
    "MISO": "MIDW",
    "NEVP": "NW",
    "NSB": "FLA",
    "NWMT": "NW",
    "NYIS": "NY",
    "OVEC": "MIDA",
    "PACE": "NW",
    "PACW": "NW",
    "PGE": "NW",
    "PJM": "MIDA",
    "PNM": "SW",
    "PSCO": "NW",
    "PSEI": "NW",
    "SC": "CAR",
    "SCEG": "CAR",
    "SCL": "NW",
    "SEC": "FLA",
    "SEPA": "SE",
    "SIKE": "MIDW",
    "SOCO": "SE",
    "SPA": "CENT",
    "SRP": "SW",
    "SWPP": "CENT",
    "SWPW": "NW",
    "TAL": "FLA",
    "TEC": "FLA",
    "TEPC": "SW",
    "TIDC": "CAL",
    "TPWR": "NW",
    "TVA": "TEN",
    "WACM": "NW",
    "WALC": "SW",
    "WAUE": "NW",
    "WAUW": "NW",
    "WWA": "NW",
    "YAD": "CAR",
    "AESO": "CAN",
    "BCHA": "CAN",
    "HQT": "CAN",
    "IESO": "CAN",
    "MHEB": "CAN",
    "NBSO": "CAN",
    "SPC": "CAN",
    "CEN": "MEX",
    "CFE": "MEX",
}

region_to_grid = {
    "CAL": "Western",
    "CAR": "Eastern",
    "CENT": "Eastern",
    "FLA": "Eastern",
    "MIDA": "Eastern",
    "MIDW": "Eastern",
    "NE": "Eastern",
    "NW": "Western",
    "NY": "Eastern",
    "SE": "Eastern",
    "SW": "Western",
    "TEN": "Eastern",
    "TEX": "Texas",
}

ba_columns_to_use = [
    "BA",
    "UTC time",
    # "Local date",
    # "Hour",
    "Local time",
    # "Time zone",
    # "Generation only?",
    # "Demand forecast",
    # "Demand",
    # "Net generation",
    # "Total interchange",
    # "Imputed demand",
    # "Imputed net generation",
    # "Imputed total interchange",
    "Adjusted demand",
    "Adjusted net generation",
    "Adjusted total interchange",
    # "NG: COL",
    # "NG: NG",
    # "NG: NUC",
    # "NG: OIL",
    # "NG: GEO",
    # "NG: WAT",
    # "NG: PS",
    # "NG: SUN",
    # "NG: SNB",
    # "NG: WND",
    # "NG: WNB",
    # "NG: BAT",
    # "NG: OES",
    # "NG: UES",
    # "NG: OTH",
    # "NG: UNK",
    # "Imputed COL Gen",
    # "Imputed NG Gen",
    # "Imputed NUC Gen",
    # "Imputed OIL Gen",
    # "Imputed GEO Gen",
    # "Imputed WAT Gen",
    # "Imputed PS Gen",
    # "Imputed SUN Gen",
    # "Imputed SNB Gen",
    # "Imputed WND Gen",
    # "Imputed WNB Gen",
    # "Imputed BAT Gen",
    # "Imputed OES Gen",
    # "Imputed UES Gen",
    # "Imputed OTH Gen",
    # "Imputed UNK Gen",
    # "Adjusted COL Gen",
    # "Adjusted NG Gen",
    # "Adjusted NUC Gen",
    # "Adjusted OIL Gen",
    # "Adjusted GEO Gen",
    # "Adjusted WAT Gen",
    # "Adjusted PS Gen",
    # "Adjusted SUN Gen",
    # "Adjusted SNB Gen",
    # "Adjusted WND Gen",
    # "Adjusted WNB Gen",
    # "Adjusted BAT Gen",
    # "Adjusted OES Gen",
    # "Adjusted UES Gen",
    # "Adjusted OTH Gen",
    # "Adjusted UNK Gen",
    # "CO2 Factor: COL",
    # "CO2 Factor: NG",
    # "CO2 Factor: OIL",
    # "CO2 Emissions: COL",
    # "CO2 Emissions: NG",
    # "CO2 Emissions: OIL",
    # "CO2 Emissions: Other",
    "CO2 Emissions Generated",
    "CO2 Emissions Imported",
    "CO2 Emissions Exported",
    "CO2 Emissions Consumed",
    "Positive Generation",
    "Consumed Electricity",
    "CO2 Emissions Intensity for Generated Electricity",
    "CO2 Emissions Intensity for Consumed Electricity",
]

region_columns_to_use = [
    "Region",
    "UTC time",
    # "Local date",
    # "Hour",
    "Local time",
    # "Time zone",
    # "Demand forecast",
    "Demand",
    "Net generation",
    "Total interchange",
    # "Sum (NG)",
    # "NG: COL",
    # "NG: NG",
    # "NG: NUC",
    # "NG: OIL",
    # "NG: GEO",
    # "NG: WAT",
    # "NG: PS",
    # "NG: SUN",
    # "NG: SNB",
    # "NG: WND",
    # "NG: WNB",
    # "NG: BAT",
    # "NG: OES",
    # "NG: UES",
    # "NG: OTH",
    # "NG: UNK",
    # "Sum (Trade)",
    # "Sum (Imports)",
    # "Sum (Exports)",
    # "Balance NG D TI",
    # "Balance TI Trade",
    # "Balance NG",
    # "CO2 Factor: COL",
    # "CO2 Factor: NG",
    # "CO2 Factor: OIL",
    # "Positive Gen from COL",
    # "Positive Gen from NG",
    # "Positive Gen from OIL",
    # "CO2 Emissions: COL",
    # "CO2 Emissions: NG",
    # "CO2 Emissions: OIL",
    # "CO2 Emissions: Other",
    "CO2 Emissions Generated",
    "CO2 Emissions Imported",
    "CO2 Emissions Exported",
    "CO2 Emissions Consumed",
    "Positive Generation",
    "Consumed Electricity",
    "CO2 Emissions Intensity for Generated Electricity",
    "CO2 Emissions Intensity for Consumed Electricity",
]

column_map = {
    "BA": "ba_code",
    "Balancing Authority": "ba_code",
    "UTC time": "datetime_utc",
    "Local time": "datetime_local",
    "Adjusted demand": "demand_mwh",
    "Adjusted net generation": "net_generation_mwh",
    "Adjusted total interchange": "total_interchange_mwh",
    "Positive Generation": "positive_generation_mwh",
    "Consumed Electricity": "consumed_mwh",
    "CO2 Emissions Intensity for Generated Electricity": "co2_rate_generated_lb_per_mwh",
    "CO2 Emissions Intensity for Consumed Electricity": "co2_rate_consumed_lb_per_mwh",
    "CO2 Emissions Generated": "co2_emissions_generated_lb",
    "CO2 Emissions Imported": "co2_emissions_imported_lb",
    "CO2 Emissions Exported": "co2_emissions_exported_lb",
    "CO2 Emissions Consumed": "co2_emissions_consumed_lb",
}

# temporary, only use MISO
ba_codes = ["MISO"]

In [47]:
# load ba data
# download all ba data into single dataframe and standardize
ba_data = []
for ba_code in ba_codes:
    try:
        b_data = pd.read_excel(
            f"https://www.eia.gov/electricity/gridmonitor/knownissues/xls/{ba_code}.xlsx",
            sheet_name="Published Hourly Data",
            usecols=lambda col: col in (ba_columns_to_use),
        )
    except ValueError:
        print(f"No data found for {ba_code}")
        continue
    # filter data to a specific local year
    b_data = b_data[b_data["Local time"].dt.year == year]
    # rename columns
    b_data.rename(columns=column_map, inplace=True)
    # convert emission rates
    for col in ["co2_rate_generated_lb_per_mwh", "co2_rate_consumed_lb_per_mwh"]:
        if col in b_data.columns:
            b_data[col] = b_data[col] * 1000
    # rename column
    b_data.rename(
        columns={
            "co2_rate_generated_lb_per_mwh": "ba_hour_generated",
            "co2_rate_consumed_lb_per_mwh": "ba_hour_consumed",
        },
        inplace=True,
    )
    # convert emissions totals from metric tons to pounds
    for col in [
        "co2_emissions_generated_lb",
        "co2_emissions_imported_lb",
        "co2_emissions_exported_lb",
        "co2_emissions_consumed_lb",
    ]:
        if col in b_data.columns:
            b_data[col] = b_data[col] * 2204.62
    ba_data.append(b_data)


ba_data = pd.concat(ba_data)
ba_data

,ba_code,datetime_utc,datetime_local,demand_mwh,net_generation_mwh,total_interchange_mwh,co2_emissions_generated_lb,co2_emissions_imported_lb,co2_emissions_exported_lb,co2_emissions_consumed_lb,positive_generation_mwh,consumed_mwh,ba_hour_generated,ba_hour_consumed
83327,MISO,2025-01-01 05:00:00,2025-01-01 00:00:00,68558.0,67617.0,-2683.0,5.716482e+07,3.437212e+06,2.038506e+06,5.856352e+07,67616.0,70299.0,845.433264,833.063365
83328,MISO,2025-01-01 06:00:00,2025-01-01 01:00:00,67121.0,65578.0,-3481.0,5.419934e+07,3.730774e+06,1.847468e+06,5.608264e+07,65580.0,69061.0,826.461400,812.074024
83329,MISO,2025-01-01 07:00:00,2025-01-01 02:00:00,65957.0,63528.0,-4156.0,5.028911e+07,4.250254e+06,1.876670e+06,5.266269e+07,63529.0,67685.0,791.592952,778.055585
83330,MISO,2025-01-01 08:00:00,2025-01-01 03:00:00,65125.0,62717.0,-4309.0,4.803617e+07,4.513170e+06,1.917258e+06,5.063208e+07,62716.0,67025.0,765.931616,755.420807
83331,MISO,2025-01-01 09:00:00,2025-01-01 04:00:00,64438.0,62367.0,-3895.0,4.692525e+07,4.377294e+06,2.069280e+06,4.923326e+07,62367.0,66262.0,752.405107,743.009008
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92082,MISO,2026-01-01 00:00:00,2025-12-31 19:00:00,79625.0,82032.0,-178.0,8.048856e+07,2.679193e+06,2.246208e+06,8.092155e+07,82033.0,82211.0,981.172964,984.315322
92083,MISO,2026-01-01 01:00:00,2025-12-31 20:00:00,79490.0,81428.0,-604.0,8.060988e+07,2.914575e+06,2.206594e+06,8.131786e+07,81429.0,82033.0,989.940668,991.282282
92084,MISO,2026-01-01 02:00:00,2025-12-31 21:00:00,78031.0,80132.0,-405.0,7.712458e+07,2.803505e+06,2.237585e+06,7.769050e+07,80147.0,80552.0,962.289059,964.476380
92085,MISO,2026-01-01 03:00:00,2025-12-31 22:00:00,76936.0,78863.0,-512.0,7.625037e+07,3.089163e+06,2.536992e+06,7.680254e+07,78864.0,79376.0,966.858979,967.578832


# Imports Analysis

In [ ]:
# download interchange data to evaluate ba imports
interchange = pd.concat(
    [
        pd.read_csv(
            "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_INTERCHANGE_2025_Jan_Jun.csv",
            usecols=[
                "Balancing Authority",
                "UTC Time at End of Hour",
                "Local Time at End of Hour",
                "Directly Interconnected Balancing Authority",
                "Interchange (MW)",
            ],
        ),
        pd.read_csv(
            "https://www.eia.gov/electricity/gridmonitor/sixMonthFiles/EIA930_INTERCHANGE_2025_Jul_Dec.csv",
            usecols=[
                "Balancing Authority",
                "UTC Time at End of Hour",
                "Local Time at End of Hour",
                "Directly Interconnected Balancing Authority",
                "Interchange (MW)",
            ],
        ),
    ],
    axis=0,
)
interchange = interchange.rename(columns=column_map)

# calculate imports and exports for each hour
interchange["imports_mw"] = (
    interchange["Interchange (MW)"].abs().where(interchange["Interchange (MW)"] <= 0, 0)
)
interchange["exports_mw"] = interchange["Interchange (MW)"].where(
    interchange["Interchange (MW)"] >= 0, 0
)

# get total annual imports
annual_imports = (
    interchange.groupby("ba_code")[["imports_mw", "exports_mw"]].sum().reset_index()
)
annual_imports

In [ ]:
import_share = (
    ba_data.groupby("ba_code")[["demand_mwh"]]
    .sum()
    .merge(annual_imports, on="ba_code", how="left")
)
import_share["import_share"] = import_share["imports_mw"] / (
    import_share["demand_mwh"] + import_share["exports_mw"]
)
import_share.sort_values("import_share", ascending=False)

,ba_code,demand_mwh,imports_mw,exports_mw,import_share
0,MISO,663821391.0,44596445.0,25641296.0,0.064683


In [ ]:
import_e_share = ba_data.groupby("ba_code")[
    [
        "co2_emissions_imported_lb",
        "co2_emissions_exported_lb",
        "co2_emissions_consumed_lb",
    ]
].sum()
import_e_share["import_share"] = import_e_share["co2_emissions_imported_lb"] / (
    import_e_share["co2_emissions_consumed_lb"]
    + import_e_share["co2_emissions_exported_lb"]
)
import_e_share.sort_values("import_share", ascending=False)


,co2_emissions_imported_lb,co2_emissions_exported_lb,co2_emissions_consumed_lb,import_share
ba_code,,,,
MISO,3.545434e+10,2.500115e+10,6.665143e+11,0.05127


# Hierarchy Analysis

In [48]:
# add category columns
ba_data["year"] = year
ba_data["month"] = ba_data["datetime_local"].dt.month
ba_data["region"] = ba_data["ba_code"].map(ba_to_region)
ba_data["grid"] = ba_data["region"].map(region_to_grid)
ba_data

,ba_code,datetime_utc,datetime_local,demand_mwh,net_generation_mwh,total_interchange_mwh,co2_emissions_generated_lb,co2_emissions_imported_lb,co2_emissions_exported_lb,co2_emissions_consumed_lb,positive_generation_mwh,consumed_mwh,ba_hour_generated,ba_hour_consumed,year,month,region,grid
83327,MISO,2025-01-01 05:00:00,2025-01-01 00:00:00,68558.0,67617.0,-2683.0,5.716482e+07,3.437212e+06,2.038506e+06,5.856352e+07,67616.0,70299.0,845.433264,833.063365,2025,1,MIDW,Eastern
83328,MISO,2025-01-01 06:00:00,2025-01-01 01:00:00,67121.0,65578.0,-3481.0,5.419934e+07,3.730774e+06,1.847468e+06,5.608264e+07,65580.0,69061.0,826.461400,812.074024,2025,1,MIDW,Eastern
83329,MISO,2025-01-01 07:00:00,2025-01-01 02:00:00,65957.0,63528.0,-4156.0,5.028911e+07,4.250254e+06,1.876670e+06,5.266269e+07,63529.0,67685.0,791.592952,778.055585,2025,1,MIDW,Eastern
83330,MISO,2025-01-01 08:00:00,2025-01-01 03:00:00,65125.0,62717.0,-4309.0,4.803617e+07,4.513170e+06,1.917258e+06,5.063208e+07,62716.0,67025.0,765.931616,755.420807,2025,1,MIDW,Eastern
83331,MISO,2025-01-01 09:00:00,2025-01-01 04:00:00,64438.0,62367.0,-3895.0,4.692525e+07,4.377294e+06,2.069280e+06,4.923326e+07,62367.0,66262.0,752.405107,743.009008,2025,1,MIDW,Eastern
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
92082,MISO,2026-01-01 00:00:00,2025-12-31 19:00:00,79625.0,82032.0,-178.0,8.048856e+07,2.679193e+06,2.246208e+06,8.092155e+07,82033.0,82211.0,981.172964,984.315322,2025,12,MIDW,Eastern
92083,MISO,2026-01-01 01:00:00,2025-12-31 20:00:00,79490.0,81428.0,-604.0,8.060988e+07,2.914575e+06,2.206594e+06,8.131786e+07,81429.0,82033.0,989.940668,991.282282,2025,12,MIDW,Eastern
92084,MISO,2026-01-01 02:00:00,2025-12-31 21:00:00,78031.0,80132.0,-405.0,7.712458e+07,2.803505e+06,2.237585e+06,7.769050e+07,80147.0,80552.0,962.289059,964.476380,2025,12,MIDW,Eastern
92085,MISO,2026-01-01 03:00:00,2025-12-31 22:00:00,76936.0,78863.0,-512.0,7.625037e+07,3.089163e+06,2.536992e+06,7.680254e+07,78864.0,79376.0,966.858979,967.578832,2025,12,MIDW,Eastern


In [ ]:
# print the number of rows with na values grouped by ba_code
ba_data[
    ba_data[
        [
            "positive_generation_mwh",
            "consumed_mwh",
            "co2_emissions_consumed_lb",
            "co2_emissions_generated_lb",
        ]
    ]
    .isna()
    .any(axis=1)
].groupby("ba_code").size()

ba_code
MISO    23
dtype: int64

In [ ]:
# calculate various aggregations of data
# drop rows with missing values
ba_data = ba_data[
    ~ba_data[
        [
            "positive_generation_mwh",
            "consumed_mwh",
            "co2_emissions_consumed_lb",
            "co2_emissions_generated_lb",
        ]
    ]
    .isna()
    .any(axis=1)
]

# ba_month_consumed
ba_data["ba_month_consumed"] = ba_data.groupby(["ba_code", "month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["ba_code", "month"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# ba_month_generated
ba_data["ba_month_generated"] = ba_data.groupby(["ba_code", "month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["ba_code", "month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# ba_year_consumed
ba_data["ba_year_consumed"] = ba_data.groupby(["ba_code", "year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["ba_code", "year"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# ba_year_generated
ba_data["ba_year_generated"] = ba_data.groupby(["ba_code", "year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["ba_code", "year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

# region_hour_consumed
ba_data["region_hour_consumed"] = ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["co2_emissions_consumed_lb"].transform("sum") / ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["consumed_mwh"].transform("sum")
# region_hour_generated
ba_data["region_hour_generated"] = ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["co2_emissions_generated_lb"].transform("sum") / ba_data.groupby(
    ["region", "datetime_utc"], dropna=False
)["positive_generation_mwh"].transform("sum")
# region_month_consumed
ba_data["region_month_consumed"] = ba_data.groupby(["region", "month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["region", "month"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# region_month_generated
ba_data["region_month_generated"] = ba_data.groupby(["region", "month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["region", "month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# region_year_consumed
ba_data["region_year_consumed"] = ba_data.groupby(["region", "year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["region", "year"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# region_year_generated
ba_data["region_year_generated"] = ba_data.groupby(["region", "year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["region", "year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

# grid_hour_consumed
ba_data["grid_hour_consumed"] = ba_data.groupby(["grid", "datetime_utc"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["grid", "datetime_utc"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# grid_hour_generated
ba_data["grid_hour_generated"] = ba_data.groupby(
    ["grid", "datetime_utc"], dropna=False
)["co2_emissions_generated_lb"].transform("sum") / ba_data.groupby(
    ["grid", "datetime_utc"], dropna=False
)["positive_generation_mwh"].transform("sum")
# grid_month_consumed
ba_data["grid_month_consumed"] = ba_data.groupby(["grid", "month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["grid", "month"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# grid_month_generated
ba_data["grid_month_generated"] = ba_data.groupby(["grid", "month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["grid", "month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# grid_year_consumed
ba_data["grid_year_consumed"] = ba_data.groupby(["grid", "year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["grid", "year"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# grid_year_generated
ba_data["grid_year_generated"] = ba_data.groupby(["grid", "year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["grid", "year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

# national_hour_consumed
ba_data["national_hour_consumed"] = ba_data.groupby(["datetime_utc"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["datetime_utc"], dropna=False)[
    "consumed_mwh"
].transform("sum")
# national_hour_generated
ba_data["national_hour_generated"] = ba_data.groupby(["datetime_utc"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["datetime_utc"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# national_month_consumed
ba_data["national_month_consumed"] = ba_data.groupby(["month"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["month"], dropna=False)["consumed_mwh"].transform(
    "sum"
)
# national_month_generated
ba_data["national_month_generated"] = ba_data.groupby(["month"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["month"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")
# national_year_consumed
ba_data["national_year_consumed"] = ba_data.groupby(["year"], dropna=False)[
    "co2_emissions_consumed_lb"
].transform("sum") / ba_data.groupby(["year"], dropna=False)["consumed_mwh"].transform(
    "sum"
)
# national_year_generated
ba_data["national_year_generated"] = ba_data.groupby(["year"], dropna=False)[
    "co2_emissions_generated_lb"
].transform("sum") / ba_data.groupby(["year"], dropna=False)[
    "positive_generation_mwh"
].transform("sum")

In [89]:
# calculate total emissions
emissions_calc = ba_data[["ba_code", "region", "datetime_utc"]].copy()
for spatial_level in ["ba", "region", "grid", "national"]:
    for temporal_level in ["hour", "month", "year"]:
        for type in ["consumed", "generated"]:
            emissions_calc[f"{spatial_level}_{temporal_level}_{type}"] = (
                ba_data[f"{spatial_level}_{temporal_level}_{type}"]
                * ba_data["demand_mwh"]
            )

# add US total to the dataframe
us_total = (
    emissions_calc.drop(columns=["ba_code"])
    .groupby(["datetime_utc"])
    .sum()
    .reset_index()
)
us_total["ba_code"] = "US"
emissions_calc = pd.concat([emissions_calc, us_total])


emissions_calc

,ba_code,region,datetime_utc,ba_hour_consumed,ba_hour_generated,ba_month_consumed,ba_month_generated,ba_year_consumed,ba_year_generated,region_hour_consumed,...,grid_month_consumed,grid_month_generated,grid_year_consumed,grid_year_generated,national_hour_consumed,national_hour_generated,national_month_consumed,national_month_generated,national_year_consumed,national_year_generated
83327,MISO,MIDW,2025-01-01 05:00:00,5.711316e+07,5.796121e+07,7.092458e+07,7.163617e+07,6.759092e+07,6.845361e+07,5.711316e+07,...,7.092458e+07,7.163617e+07,6.759092e+07,6.845361e+07,5.711316e+07,5.796121e+07,7.092458e+07,7.163617e+07,6.759092e+07,6.845361e+07
83328,MISO,MIDW,2025-01-01 06:00:00,5.450722e+07,5.547292e+07,6.943797e+07,7.013465e+07,6.617419e+07,6.701880e+07,5.450722e+07,...,6.943797e+07,7.013465e+07,6.617419e+07,6.701880e+07,5.450722e+07,5.547292e+07,6.943797e+07,7.013465e+07,6.617419e+07,6.701880e+07
83329,MISO,MIDW,2025-01-01 07:00:00,5.131821e+07,5.221110e+07,6.823379e+07,6.891839e+07,6.502661e+07,6.585657e+07,5.131821e+07,...,6.823379e+07,6.891839e+07,6.502661e+07,6.585657e+07,5.131821e+07,5.221110e+07,6.823379e+07,6.891839e+07,6.502661e+07,6.585657e+07
83330,MISO,MIDW,2025-01-01 08:00:00,4.919678e+07,4.988130e+07,6.737307e+07,6.804903e+07,6.420635e+07,6.502583e+07,4.919678e+07,...,6.737307e+07,6.804903e+07,6.420635e+07,6.502583e+07,4.919678e+07,4.988130e+07,6.737307e+07,6.804903e+07,6.420635e+07,6.502583e+07
83331,MISO,MIDW,2025-01-01 09:00:00,4.787801e+07,4.848348e+07,6.666236e+07,6.733119e+07,6.352904e+07,6.433988e+07,4.787801e+07,...,6.666236e+07,6.733119e+07,6.352904e+07,6.433988e+07,4.787801e+07,4.848348e+07,6.666236e+07,6.733119e+07,6.352904e+07,6.433988e+07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8732,US,MIDW,2026-01-01 00:00:00,7.837611e+07,7.812590e+07,8.021718e+07,8.061152e+07,7.850181e+07,7.950376e+07,7.837611e+07,...,8.021718e+07,8.061152e+07,7.850181e+07,7.950376e+07,7.837611e+07,7.812590e+07,8.021718e+07,8.061152e+07,7.850181e+07,7.950376e+07
8733,US,MIDW,2026-01-01 01:00:00,7.879703e+07,7.869038e+07,8.008118e+07,8.047485e+07,7.836871e+07,7.936896e+07,7.879703e+07,...,8.008118e+07,8.047485e+07,7.836871e+07,7.936896e+07,7.879703e+07,7.869038e+07,8.008118e+07,8.047485e+07,7.836871e+07,7.936896e+07
8734,US,MIDW,2026-01-01 02:00:00,7.525906e+07,7.508838e+07,7.861133e+07,7.899777e+07,7.693029e+07,7.791218e+07,7.525906e+07,...,7.861133e+07,7.899777e+07,7.693029e+07,7.791218e+07,7.525906e+07,7.508838e+07,7.861133e+07,7.899777e+07,7.693029e+07,7.791218e+07
8735,US,MIDW,2026-01-01 03:00:00,7.444165e+07,7.438626e+07,7.750818e+07,7.788920e+07,7.585074e+07,7.681885e+07,7.444165e+07,...,7.750818e+07,7.788920e+07,7.585074e+07,7.681885e+07,7.444165e+07,7.438626e+07,7.750818e+07,7.788920e+07,7.585074e+07,7.681885e+07


In [90]:
# calculate the annual error metrics for each ba and for the US total
annual_error_wide = (
    emissions_calc.drop(columns=["datetime_utc"]).groupby(["ba_code"]).sum()
)
annual_error = []
for ba in annual_error_wide.index:
    for spatial_level in ["ba", "grid", "national"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_error.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": (
                            annual_error_wide.loc[
                                ba, f"{spatial_level}_{temporal_level}_{type}"
                            ]
                            - annual_error_wide.loc[ba, f"ba_hour_consumed"]
                        )
                        / annual_error_wide.loc[ba, f"ba_hour_consumed"],
                    }
                )

annual_error = pd.DataFrame(annual_error)
annual_error


,ba_code,spatial_level,temporal_level,type,error
0,MISO,ba,hour,consumed,0.000000
1,MISO,ba,hour,generated,0.013652
2,MISO,ba,month,consumed,-0.000227
3,MISO,ba,month,generated,0.012655
4,MISO,ba,year,consumed,-0.000287
5,MISO,ba,year,generated,0.012473
6,MISO,grid,hour,consumed,0.000000
7,MISO,grid,hour,generated,0.013652
8,MISO,grid,month,consumed,-0.000227
9,MISO,grid,month,generated,0.012655


In [82]:
# calculate hourly MAPE
hourly_mape = emissions_calc.copy()
for spatial_level in ["ba", "grid", "national"]:
    for temporal_level in ["hour", "month", "year"]:
        for type in ["consumed", "generated"]:
            if f"{spatial_level}_{temporal_level}_{type}" == "ba_hour_consumed":
                continue
            else:
                hourly_mape[f"{spatial_level}_{temporal_level}_{type}"] = (
                    (
                        hourly_mape[f"{spatial_level}_{temporal_level}_{type}"]
                        - hourly_mape[f"ba_hour_consumed"]
                    )
                    / hourly_mape[f"ba_hour_consumed"]
                ).abs()
hourly_mape["ba_hour_consumed"] = 0.0
annual_mape_wide = (
    hourly_mape.drop(columns=["datetime_utc"]).groupby(["ba_code"]).mean()
)

annual_mape = []
for ba in annual_mape_wide.index:
    for spatial_level in ["ba", "grid", "national"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_mape.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": annual_mape_wide.loc[
                            ba, f"{spatial_level}_{temporal_level}_{type}"
                        ],
                    }
                )

annual_mape = pd.DataFrame(annual_mape)
annual_mape

,ba_code,spatial_level,temporal_level,type,error
0,MISO,ba,hour,consumed,0.000000e+00
1,MISO,ba,hour,generated,1.391240e-02
2,MISO,ba,month,consumed,9.641972e-02
3,MISO,ba,month,generated,9.786438e-02
4,MISO,ba,year,consumed,1.161340e-01
5,MISO,ba,year,generated,1.175144e-01
6,MISO,grid,hour,consumed,4.133139e-17
7,MISO,grid,hour,generated,1.391240e-02
8,MISO,grid,month,consumed,9.641972e-02
9,MISO,grid,month,generated,9.786438e-02


# Local regions
Do "local" subdivisions provide meaningful gain in accuracy. 

Test 1: MISO
- Use data from MISO dashboard for LRZs, and LRZ demand data from EIA-930
- Use that as the source of truth, and compare to MISO-wide numbers

Test 2: EIA "regions" as proxy for large regions
- Examples: Northwest, Florida, Southwest, Carolinas
- multiply BA demand by regional EFs

## EIA Proxy

In [ ]:
# EIA proxies
region_proxy = emissions_calc[
    emissions_calc["region"].isin(["FLA", "CAR", "SW", "NW"])
].copy()

# calculate the annual error metrics for each ba and for the US total
annual_error_wide = (
    emissions_calc.drop(columns=["datetime_utc"]).groupby(["ba_code"]).sum()
)
annual_error = []
for ba in annual_error_wide.index:
    for spatial_level in ["ba", "region"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_error.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": (
                            annual_error_wide.loc[
                                ba, f"{spatial_level}_{temporal_level}_{type}"
                            ]
                            - annual_error_wide.loc[ba, f"ba_hour_consumed"]
                        )
                        / annual_error_wide.loc[ba, f"ba_hour_consumed"],
                    }
                )

annual_error = pd.DataFrame(annual_error)
annual_error


,ba_code,spatial_level,temporal_level,type,error
0,MISO,ba,hour,consumed,0.000000
1,MISO,ba,hour,generated,0.013652
2,MISO,ba,month,consumed,-0.000227
3,MISO,ba,month,generated,0.012655
4,MISO,ba,year,consumed,-0.000287
5,MISO,ba,year,generated,0.012473
6,MISO,region,hour,consumed,0.000000
7,MISO,region,hour,generated,0.013652
8,MISO,region,month,consumed,-0.000227
9,MISO,region,month,generated,0.012655


In [93]:
# calculate hourly MAPE
hourly_region_mape = region_proxy.copy()
for spatial_level in ["ba", "region"]:
    for temporal_level in ["hour", "month", "year"]:
        for type in ["consumed", "generated"]:
            if f"{spatial_level}_{temporal_level}_{type}" == "ba_hour_consumed":
                continue
            else:
                hourly_region_mape[f"{spatial_level}_{temporal_level}_{type}"] = (
                    (
                        hourly_region_mape[f"{spatial_level}_{temporal_level}_{type}"]
                        - hourly_region_mape[f"ba_hour_consumed"]
                    )
                    / hourly_region_mape[f"ba_hour_consumed"]
                ).abs()
hourly_region_mape["ba_hour_consumed"] = 0.0
annual_region_mape_wide = (
    hourly_region_mape.drop(columns=["datetime_utc"]).groupby(["ba_code"]).mean()
)

annual_region_mape = []
for ba in annual_region_mape_wide.index:
    for spatial_level in ["ba", "region"]:
        for temporal_level in ["hour", "month", "year"]:
            for type in ["consumed", "generated"]:
                annual_region_mape.append(
                    {
                        "ba_code": ba,
                        "spatial_level": spatial_level,
                        "temporal_level": temporal_level,
                        "type": type,
                        "error": annual_region_mape_wide.loc[
                            ba, f"{spatial_level}_{temporal_level}_{type}"
                        ],
                    }
                )

annual_region_mape = pd.DataFrame(annual_region_mape)
annual_region_mape

""


## MISO data
Get subregion demand data from 930

In [ ]:
# get subregion demand data from 930
subregion_cols = {
    "Subregion 0001": "1",
    "Subregion 0004": "4",
    "Subregion 0006": "6",
    "Subregion 0027": "2/7",
    "Subregion 0035": "3/5",
    "Subregion 8910": "8/9/10",
}
miso_subregion = pd.read_excel(
    f"https://www.eia.gov/electricity/gridmonitor/knownissues/xls/MISO.xlsx",
    sheet_name="Published Hourly Data",
    usecols=lambda col: col in (ba_columns_to_use + list(subregion_cols.keys())),
)
# filter data to a specific local year
miso_subregion = miso_subregion[miso_subregion["Local time"].dt.year == year]
# rename columns
miso_subregion.rename(columns=column_map, inplace=True)
miso_subregion.rename(columns=subregion_cols, inplace=True)
# pivot the data so subregion is a column, and demand is the value
miso_subregion = miso_subregion.melt(
    id_vars=["datetime_utc"],
    value_vars=list(subregion_cols.values()),
    var_name="subregion",
    value_name="demand",
)
miso_subregion["datetime_utc"] = pd.to_datetime(miso_subregion["datetime_utc"], utc=True)
miso_subregion

,datetime_utc,subregion,demand
0,2025-01-01 05:00:00,1,11113.0
1,2025-01-01 06:00:00,1,10837.0
2,2025-01-01 07:00:00,1,10565.0
3,2025-01-01 08:00:00,1,10339.0
4,2025-01-01 09:00:00,1,10186.0
...,...,...,...
52555,2026-01-01 00:00:00,8/9/10,19525.0
52556,2026-01-01 01:00:00,8/9/10,20139.0
52557,2026-01-01 02:00:00,8/9/10,20095.0
52558,2026-01-01 03:00:00,8/9/10,20023.0


In [ ]:
# MISO consumed emission rates (same source as https://miso.singularity.energy/consumed)

API_KEY = "653f70e4840e47ab858bae9a053febad"  # paste your key here

# Same endpoint the consumed dashboard uses:
# https://miso.singularity.energy/consumed
# GET https://miso-v2.api.singularity.energy/api/v1/carbonflube/consumed-events
# Rates are always returned. Totals are omitted when confidential (recent LRZ/LBA).
ENDPOINT = "https://miso-v2.api.singularity.energy/api/v1/carbonflube/consumed-events"

# Query window in UTC. `start` is inclusive, `end` is exclusive.
start = "2025-01-01T00:00:00Z"
end = "2025-01-02T00:00:00Z"

agg_type = "lrz"
# keys: LRZ id(s) to query; values: whether to combine those ids into one event
agg_ids_combine = {
    "1": False,
    "2,7": True,
    "3,5": True,
    "4": False,
    "6": False,
    "8,9,10": True,
}
units = "us"  # "metric" (kg CO2e) or "us" (lbs CO2e)
chunk_days = 7  # API responses stay small; increase if queries are reliably fast


def _utc(ts):
    ts = pd.Timestamp(ts)
    return ts.tz_localize("UTC") if ts.tzinfo is None else ts.tz_convert("UTC")


def fetch_consumed_rates(
    start, end, agg_type="lrz", agg_ids=None, combine=False, units="us"
):
    """Query dashboard consumed-events for emission rates, chunking the date range."""
    start_ts, end_ts = _utc(start), _utc(end)
    events = []
    chunk_start = start_ts
    while chunk_start < end_ts:
        chunk_end = min(chunk_start + pd.Timedelta(days=chunk_days), end_ts)
        params = {
            "start": chunk_start.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "end": chunk_end.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "agg_type": agg_type,
            "units": units,
        }
        if agg_ids:
            params["agg_ids"] = agg_ids
        if combine:
            params["combine"] = "true"

        headers = {"User-Agent": "open-grid-emissions-notebook"}
        if API_KEY:
            headers["x-api-key"] = API_KEY

        response = requests.get(ENDPOINT, headers=headers, params=params, timeout=60)
        if not response.ok:
            raise RuntimeError(
                f"{response.status_code} {response.reason} for "
                f"{params['start']}–{params['end']} agg_ids={agg_ids}: {response.text}"
            )
        events.extend(response.json().get("data", []))
        chunk_start = chunk_end

    if not events:
        return pd.DataFrame()

    df = pd.json_normalize(events)
    df.columns = [c.replace("emissions.", "") for c in df.columns]
    # Keep rates only; drop totals (confidential for some LRZ queries) and fuel mix.
    drop_cols = [
        c for c in df.columns if c.startswith("total_") or c.startswith("fuel_mix.")
    ]
    df = df.drop(columns=drop_cols, errors="ignore")
    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df["requested_agg_ids"] = agg_ids
    df["combine"] = combine
    return df


frames = []
for agg_ids, combine in agg_ids_combine.items():
    frames.append(
        fetch_consumed_rates(
            start=start,
            end=end,
            agg_type=agg_type,
            agg_ids=agg_ids,
            combine=combine,
            units=units,
        )
    )

lrz_emissions = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["timestamp", "requested_agg_ids", "agg_id"])
    .reset_index(drop=True)
)
lrz_emissions.rename(
    columns={
        "timestamp": "datetime_utc",
        "agg_id": "subregion",
        "rate_co2e_lbs_per_mwh": "local_hourly_consumed",
    },
    inplace=True,
)
lrz_emissions["subregion"] = lrz_emissions["subregion"].map(
    {"1": "1", "4": "4", "6": "6", "2,7": "2/7", "3,5": "3/5", "10,8,9": "8/9/10"}
)
lrz_emissions = lrz_emissions[
    ["subregion", "datetime_utc", "local_hourly_consumed"]
].sort_values(["subregion", "datetime_utc"])

# get miso emissions
miso_emissions = fetch_consumed_rates(
    start=start,
    end=end,
    agg_type="miso",
    agg_ids="miso",
    combine=False,
    units=units,
)
miso_emissions.rename(
    columns={
        "timestamp": "datetime_utc",
        "agg_id": "subregion",
        "rate_co2e_lbs_per_mwh": "ba_hourly_consumed",
    },
    inplace=True,
)
miso_emissions = miso_emissions[["datetime_utc", "ba_hourly_consumed"]].sort_values(
    ["datetime_utc"]
)

# merge into lrz emissions
lrz_emissions = lrz_emissions.merge(
    miso_emissions, on="datetime_utc", how="left", validate="m:1"
)

lrz_emissions

,subregion,datetime_utc,local_hourly_consumed,ba_hourly_consumed
0,1,2025-01-01 00:00:00+00:00,671.12,775.10
1,1,2025-01-01 01:00:00+00:00,626.19,767.40
2,1,2025-01-01 02:00:00+00:00,579.84,768.73
3,1,2025-01-01 03:00:00+00:00,597.59,780.22
4,1,2025-01-01 04:00:00+00:00,658.17,778.44
...,...,...,...,...
145,8/9/10,2025-01-01 20:00:00+00:00,581.76,765.30
146,8/9/10,2025-01-01 21:00:00+00:00,586.65,788.22
147,8/9/10,2025-01-01 22:00:00+00:00,700.20,878.03
148,8/9/10,2025-01-01 23:00:00+00:00,762.36,931.14


In [126]:
miso_subregion.merge(lrz_emissions, on=["subregion","datetime_utc"], how="inner", validate="m:1")

,datetime_utc,subregion,demand,local_hourly_consumed,ba_hourly_consumed
0,2025-01-01 05:00:00+00:00,1,11113.0,653.41,759.91
1,2025-01-01 06:00:00+00:00,1,10837.0,693.28,738.15
2,2025-01-01 07:00:00+00:00,1,10565.0,669.03,719.08
3,2025-01-01 08:00:00+00:00,1,10339.0,648.01,713.92
4,2025-01-01 09:00:00+00:00,1,10186.0,642.42,725.28
...,...,...,...,...,...
115,2025-01-01 20:00:00+00:00,8/9/10,17973.0,581.76,765.30
116,2025-01-01 21:00:00+00:00,8/9/10,17645.0,586.65,788.22
117,2025-01-01 22:00:00+00:00,8/9/10,17548.0,700.20,878.03
118,2025-01-01 23:00:00+00:00,8/9/10,17890.0,762.36,931.14
